This file is used to ...

## CONFIGURATION

In [ ]:
CONFIG = { # CONFIG[""]
    "frames_path": "data/metadata/videos_frames.npy",
    "prompt_ids_path": "data/metadata/video_descriptions_en_short_prompt_ids.pt",

    "seed": 42,
    "BATCH_SIZE": 32, 
    "use_8bit_adam": True,
    "learning_rate": 3e-5,
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_weight_decay": 1e-2,
    "adam_epsilon": 1e-08,

    "gradient_accumulation_steps": 1,
    "mixed_precision": "no", # "no", "fp16", "bf16"

    "pretrained_model_path": "CompVis/stable-diffusion-v1-4",
    "trainable_modules": [
        "attn1.to_q",
        "attn2.to_q",
        "attn_temp",
    ],
    "enable_xformers_memory_efficient_attention": True, 
    "gradient_checkpointing": True,

    "train_data"
    "num_train_epochs": 300,
    "lr_warmup_steps"
    "gradient_accumulation_steps"

    "validation_steps": 100,
    "checkpointing_steps": 100,


    "train_data": {
        "video_path": "./data/all_40_video/1.mp4", # 路径中的文件名仅作占位符，实际由代码动态生成
        "prompt_path": "./data/all_40_video/prompts.txt",
        "n_sample_frames": 8,
        "width": 512,
        "height": 512,
        "sample_start_idx": 0,
        "sample_frame_rate": 2,
        "train_batch_size": 1,
    },
    "validation_data": {
        "prompts": [
            "a teddy bear is playing the guitar",
            "a panda is surfing on the sea"
        ],
        "video_length": 8,
        "width": 512,
        "height": 512,
        "num_inference_steps": 25,
        "guidance_scale": 7.5,
    },
    "learning_rate": 3e-5,
    "num_train_epochs": 300,
    "trainable_modules": [
        "attn1.to_q",
        "attn2.to_q",
        "attn_temp",
    ],
    "seed": 42,
    "mixed_precision": "no", # "no", "fp16", "bf16"
    "use_8bit_adam": True,
    "gradient_accumulation_steps": 1,
    "gradient_checkpointing": True,
    "enable_xformers_memory_efficient_attention": True,
    
    # 优化器参数
    "adam_beta1": 0.9,
    "adam_beta2": 0.999,
    "adam_weight_decay": 1e-2,
    "adam_epsilon": 1e-08,
    
    # LR 调度器参数
    "lr_scheduler": "constant",
    "lr_warmup_steps": 0,
    
    # 验证与保存
    "validation_steps": 100,
    "checkpointing_steps": 100,
    "max_grad_norm": 1.0,
}

print("✅  配置参数加载完成。")

✅  配置参数加载完成。


## 导入与环境设置

In [ ]:
# ==================================
# Cell 1: 导入与环境设置
# ==================================
import os
import math
from typing import Dict, Tuple

import numpy as np
import torch
import torch.nn.functional as F
import torch.utils.checkpoint
import transformers
import diffusers
from accelerate import Accelerator
from accelerate.utils import set_seed
from diffusers import AutoencoderKL, DDPMScheduler, DDIMScheduler
from diffusers.optimization import get_scheduler
from diffusers.utils.import_utils import is_xformers_available
from einops import rearrange
from omegaconf import OmegaConf, DictConfig
from tqdm.auto import tqdm
from transformers import CLIPTextModel, CLIPTokenizer

# 假设您的项目结构如下：
# project/
# |- train.ipynb
# |- models/
# |  |- unet.py
# |  |- pipeline_tuneavideo.py
# |- utils/
# |  |- dataset.py
# |  |- util.py
# |- configs/
# |- data/

from models.unet import UNet3DConditionModel
from models.pipeline_tuneavideo import TuneAVideoPipeline
from utils.util import save_videos_grid

# 设置 PyTorch CUDA 内存分配策略，以减少碎片
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:24"

print("✅ 导入与环境设置完成。")

## Define data

In [ ]:
from torch.utils.data import Dataset

class TuneMultiVideoDataset(Dataset):
    def __init__(self): 
        self.frames = np.load(CONFIG["frames_path"]) # (250, 12, 3, 288, 512)
        self.prompt_ids = torch.load(CONFIG["prompt_ids_path"]) # (250, 77)

    def __len__(self):
        return len(self.frames)

    def __getitem__(self, index):
        # # 将像素值归一化到 [-1, 1]
        norm_frames = (self.frames[index] / 127.5) - 1.0
        return norm_frames, self.prompt_ids[index]

def prepare_dataloader():
    train_dataset = TuneMultiVideoDataset()
    dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=CONFIG["BATCH_SIZE"], shuffle=True)
    return dataloader

## Load Model

In [ ]:
def load_models(model_repo_id: str, trainable_modules: Tuple[str, ...]) -> Tuple:
    """
    从 Hugging Face Hub 加载所有必要的模型，并设置可训练参数。

    Args:
        model_repo_id (str): Hugging Face Hub 上的预训练模型库 ID。
        trainable_modules (Tuple[str, ...]): 需要解冻并训练的 UNet 模块名称后缀。

    Returns:
        一个包含 (tokenizer, text_encoder, vae, unet, noise_scheduler) 的元组。
    """
    print(f"正在从 Hugging Face Hub '{model_repo_id}' 加载模型...")
    
    # from_pretrained 方法本身就支持从 Hub ID 或本地路径加载
    # 当传入的是 "runwayml/stable-diffusion-v1-5" 这样的 ID 时，它会自动从 Hub 下载
    
    try:
        noise_scheduler = DDPMScheduler.from_pretrained(model_repo_id, subfolder="scheduler")
        tokenizer = CLIPTokenizer.from_pretrained(model_repo_id, subfolder="tokenizer")
        text_encoder = CLIPTextModel.from_pretrained(model_repo_id, subfolder="text_encoder")
        vae = AutoencoderKL.from_pretrained(model_repo_id, subfolder="vae")
        
        # 特别注意：UNet3DConditionModel.from_pretrained_2d 是自定义的方法
        # 它内部会调用 diffusers 的 from_pretrained，因此同样支持 Hub ID
        unet = UNet3DConditionModel.from_pretrained_2d(model_repo_id, subfolder="unet")

    except Exception as e:
        print(f"错误: 无法从 Hugging Face Hub 加载模型 '{model_repo_id}'。")
        print("请检查模型 ID 是否正确，以及您的网络连接。")
        raise e

    # 冻结不需要训练的模型和参数
    vae.requires_grad_(False)
    text_encoder.requires_grad_(False)
    unet.requires_grad_(False)

    # 只解冻指定的可训练模块
    print("正在解冻以下模块进行微调...")
    for name, module in unet.named_modules():
        if name.endswith(trainable_modules):
            for params in module.parameters():
                params.requires_grad = True
            print(f"  - {name}")
    
    return tokenizer, text_encoder, vae, unet, noise_scheduler

## Define optimizer

In [ ]:
def create_optimizer(unet: UNet3DConditionModel) -> torch.optim.Optimizer:
    optimizer_cls = torch.optim.AdamW
    if CONFIG["use_8bit_adam"]:
        try:
            import bitsandbytes as bnb
            optimizer_cls = bnb.optim.AdamW8bit
        except ImportError:
            raise ImportError("要使用 8-bit Adam, 请安装: `pip install bitsandbytes`")
    return optimizer_cls(
        filter(lambda p: p.requires_grad, unet.parameters()),
        lr=config.learning_rate, betas=(CONFIG["adam_beta1"], CONFIG["adam_beta2"]),
        weight_decay=CONFIG["adam_weight_decay"], eps=CONFIG["adam_epsilon"]
    )

## validate and save

In [ ]:
def validate_and_save_checkpoint(epoch, accelerator, unet, vae, text_encoder, tokenizer, noise_scheduler, config):
    if accelerator.is_main_process:
        print(f"\n为周期 {epoch + 1} 运行验证...")
        unet_for_pipeline = accelerator.unwrap_model(unet)
        pipeline = TuneAVideoPipeline(
            vae=vae, text_encoder=text_encoder, tokenizer=tokenizer, unet=unet_for_pipeline,
            scheduler=DDIMScheduler.from_config(noise_scheduler.config)
        )
        pipeline.enable_vae_slicing()
        generator = torch.Generator(device=accelerator.device).manual_seed(config.seed)
        
        for i, prompt in enumerate(config.validation_data.prompts):
            with torch.autocast("cuda"):
                video = pipeline(prompt, generator=generator, **config.validation_data).videos
            safe_prompt = "".join(c for c in prompt if c.isalnum() or c in " _-")[:30]
            save_path = f"{config.output_dir}/samples/epoch-{epoch+1:04d}_prompt-{i:02d}_{safe_prompt}.gif"
            save_videos_grid(video, save_path)
        
        print(f"已为周期 {epoch+1} 保存验证样本。")
        
        if (epoch + 1) % config.checkpointing_steps == 0:
            checkpoint_dir = os.path.join(config.output_dir, f"checkpoint-{epoch+1}")
            pipeline.save_pretrained(checkpoint_dir)
            print(f"检查点已保存至 {checkpoint_dir}")

## Define train function

In [ ]:
def train_one_epoch(unet, vae, text_encoder, noise_scheduler, optimizer, lr_scheduler, train_dataloader, accelerator, weight_dtype):
    unet.train()
    for batch in train_dataloader:
        with accelerator.accumulate(unet):
            with torch.no_grad():
                pixel_values = batch["pixel_values"].to(weight_dtype)
                video_length = pixel_values.shape[1]
                pixel_values_flat = rearrange(pixel_values, "b f c h w -> (b f) c h w")
                latents = vae.encode(pixel_values_flat).latent_dist.sample()
                latents = rearrange(latents, "(b f) c h w -> b c f h w", f=video_length) * vae.config.scaling_factor
            
            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=latents.device).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            
            with torch.no_grad():
                encoder_hidden_states = text_encoder(batch["prompt_ids"])[0]
            
            if noise_scheduler.config.prediction_type == "epsilon":
                target = noise
            else: # v_prediction
                target = noise_scheduler.get_velocity(latents, noise, timesteps)

            model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
            loss = F.mse_loss(model_pred.float(), target.float(), reduction="mean")
            
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(filter(lambda p: p.requires_grad, unet.parameters()), config.max_grad_norm)
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
    return loss.detach().item()

In [ ]:
# 1. 初始化 Accelerator
accelerator = Accelerator(
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    mixed_precision=CONFIG["mixed_precision"],
)

# 2. 设置随机种子
if config.seed is not None:
    set_seed(CONFIG["seed"])

# 4. 加载模型
tokenizer, text_encoder, vae, unet, noise_scheduler = load_models(
    CONFIG["pretrained_model_path"], tuple(CONFIG["trainable_modules"])
)

# 5. 应用优化
if CONFIG["enable_xformers_memory_efficient_attention"]:
    if is_xformers_available():
        unet.enable_xformers_memory_efficient_attention()
    else:
        print("警告: xformers 不可用。")
if CONFIG["gradient_checkpointing"]:
    unet.enable_gradient_checkpointing()

# 6. 准备数据、优化器和调度器
selected_indices = get_training_indices()
train_dataloader = prepare_dataloader(CONFIG["train_data"], tokenizer, selected_indices)
optimizer = create_optimizer(unet, config)

num_update_steps_per_epoch = math.ceil(len(train_dataloader) / CONFIG["gradient_accumulation_steps"])
num_training_steps = CONFIG["num_train_epochs"] * num_update_steps_per_epoch
lr_scheduler = get_scheduler(
    config.lr_scheduler, optimizer=optimizer,
    num_warmup_steps=CONFIG["lr_warmup_steps"] * CONFIG["gradient_accumulation_steps"],
    num_training_steps=num_training_steps * accelerator.num_processes,
)

# 7. 使用 Accelerator 准备所有组件
unet, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
    unet, optimizer, train_dataloader, lr_scheduler
)

# 8. 设置数据类型并移动模型到设备
weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
text_encoder.to(accelerator.device, dtype=weight_dtype)
vae.to(accelerator.device, dtype=weight_dtype)

print("✅ 初始化完成，准备开始训练。")

In [ ]:
if accelerator.is_main_process:
    print("\n***** 开始训练 *****")
    print(f"  总周期数 = {CONFIG["num_train_epochs"]}")
    print(f"  样本数量 = {len(train_dataloader.dataset)}")
    print(f"  每个设备的批次大小 = {CONFIG["BATCH_SIZE"]}")
    print(f"  梯度累积步数 = {CONFIG["gradient_accumulation_steps"]}\n")

for epoch in range(CONFIG["num_train_epochs"]):
    progress_bar = tqdm(
        range(len(train_dataloader)),
        disable=not accelerator.is_local_main_process,
        desc=f"Epoch {epoch + 1}/{CONFIG["num_train_epochs"]}"
    )
    
    loss = train_one_epoch(
        unet, vae, text_encoder, noise_scheduler, optimizer,
        lr_scheduler, train_dataloader, accelerator, weight_dtype
    )
    
    progress_bar.set_postfix(loss=loss, lr=lr_scheduler.get_last_lr()[0])
    
    if (epoch + 1) % config.validation_steps == 0:
        validate_and_save_checkpoint(
            epoch, accelerator, unet, vae, text_encoder, tokenizer,
            noise_scheduler, config
        )

print("\n✅ Cell 5: 训练循环完成。")

In [ ]:
# ==================================
# Cell 6: 最终保存
# ==================================

accelerator.wait_for_everyone()
if accelerator.is_main_process:
    unet = accelerator.unwrap_model(unet)
    pipeline = TuneAVideoPipeline.from_pretrained(
        CONFIG["pretrained_model_path"],
        text_encoder=text_encoder,
        vae=vae,
        unet=unet,
    )
    pipeline.save_pretrained(CONFIG["output_dir"])
    print(f"\n🎉 训练完成！最终模型已保存至: {CONFIG["output_dir"]}")